In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
df=pd.read_csv("/content/returns_sustainability_dataset.csv")
df.head()

,Order_ID,Product_ID,User_ID,Order_Date,Product_Category,Product_Price,Order_Quantity,Discount_Applied,Shipping_Method,Payment_Method,...,Return_Status,Return_Reason,Days_to_Return,Order_Value,Return_Cost,Profit_Loss,CO2_Emissions,Packaging_Waste,CO2_Saved,Waste_Avoided
0,ORD00000,PROD0169,USER0195,2022-01-14,Clothing,1720.71,2,30.46,Next-Day,Wallet,...,Not Returned,No Return,0,2393.163468,0,2393.163468,2.0,0.4,2.0,0.4
1,ORD00001,PROD0318,USER1469,2022-01-03,Toys,744.06,5,29.62,Next-Day,Wallet,...,Returned,Size Issue,12,2618.347140,200,2418.347140,2.0,1.0,0.0,0.0
2,ORD00002,PROD0427,USER1812,2025-03-16,Clothing,983.68,5,47.80,Express,Wallet,...,Not Returned,No Return,0,2567.404800,0,2567.404800,1.5,1.0,1.5,1.0
3,ORD00003,PROD0323,USER1274,2024-11-06,Books,1855.65,2,2.90,Express,COD,...,Not Returned,No Return,0,3603.672300,0,3603.672300,1.5,0.4,1.5,0.4
4,ORD00004,PROD0325,USER0551,2023-06-07,Home Appliances,1770.97,5,44.42,Express,COD,...,Returned,Size Issue,11,4921.525630,200,4721.525630,1.5,1.0,0.0,0.0


In [4]:
df["Total_Value"]=df["Product_Price"] * df["Order_Quantity"]

In [5]:
df.fillna(0,inplace=True)

In [23]:
categorical_cols = ["Product_Category", "Shipping_Method", "Payment_Method", "Return_Status", "Return_Reason"]
# Filter for columns that actually exist in the DataFrame
cols_to_encode = [col for col in categorical_cols if col in df.columns]

if cols_to_encode:
    df = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)
else:
    print("No specified categorical columns to encode, or they are already encoded.")

No specified categorical columns to encode, or they are already encoded.


In [14]:
from sklearn.model_selection import train_test_split

X = df.drop(["Order_ID", "Product_ID", "User_ID", "Order_Date", "Return_Cost"], axis=1)
y = df["Return_Cost"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [27]:
X = df.drop('Return_Cost', axis=1)
y = df['Return_Cost']

In [28]:
X = pd.get_dummies(X, drop_first=True)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [30]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [31]:
pred = model.predict(X_test)

In [32]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       698
         200       1.00      1.00      1.00       302

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [33]:
df["Return_Probability"] = model.predict_proba(X)[:,1]

In [34]:
high_risk = df[df["Return_Probability"] > 0.7]

high_risk.to_csv("high_risk_products.csv", index=False)